我来分析图中的MADE架构并复现代码，同时给出思维指导。全部通过 ✓ 共 **565,205** 个参数。下面是完整的思维指导。

------

## 复现思维指导

### 核心思路：从数据流出发，逐级拆解

```
Raw I/Q [B,2,256]
  ↓ (1) 预处理
Patches [B,8,2,32]
  ↓ (2) 特征提取（二选一）
Features [B,8,128]
  ↓ (3) 掩码 + 噪声注入
Corrupted Features [B,8,128]
  ↓ (4) MLP编码器 → 位置编码 → Transformer
Contextual Embeddings [B,8,128]
  ↓ (5) 双路输出头
  ├─ 分类 logits [B, num_classes]
  └─ 重建信号 [B, 2, 256]
```

---

### 七个关键设计决策

**① 为什么要 Patch？**  
将 256 长度的 I/Q 信号切成 8 个 patch（每个 32 点），让 Transformer 处理序列而非单点，天然捕捉时序上的长程依赖，同时大幅降低注意力的计算复杂度（O(N²) 中 N=8 而非 256）。

**② IQ vs Cyclo 模式的本质区别**  
IQ 模式直接对 patch 做线性投影；Cyclo 模式用两个平行分支（归一化信号 + 原始信号）分别提取特征后拼接融合，捕捉循环稳态统计量（适合解调制信号），二者只有特征提取层不同，后续结构完全相同。

**③ 掩码 + 去噪是预训练核心**  
这是 MAE（Masked Autoencoder）思想的迁移：训练时随机掩码 30% 的 patch 并加高斯噪声，强迫网络从少量可见 patch 重建完整信号，从而学到对噪声和遮挡鲁棒的表征。微调阶段可关闭（`model.eval()` 不注入噪声）。

**④ MLP 编码器 + Transformer 的分工**  
MLP 编码器（含 BatchNorm）先做 per-patch 的局部特征变换；Transformer 再做跨 patch 的全局上下文建模——类似 ViT 的设计哲学。

**⑤ 位置编码不可缺失**  
Transformer 本身对序列顺序无感，正弦位置编码将 patch 的时序位置注入到表征中，让模型理解"第 3 段"与"第 7 段"的不同。

**⑥ 注意力池化 vs 平均池化**  
注意力池化（Attention Pooling）让模型学会"哪些 patch 对识别设备更重要"，输出的 `device_embedding` 是设备的压缩指纹，比平均池化效果更好。

**⑦ 双输出头 + 联合损失**  
```
L_total = L_CE(分类) + λ · L_MSE(重建)
```
重建 loss 约束网络保留信号的细粒度物理特征（RF指纹），分类 loss 驱动学习判别性表征，二者相互正则化。`λ` 是需要调的超参数（通常取 0.5 ~ 2.0）。

---

### 开集检测（Known/Unknown Detection）

Device Registry 本质是最近邻检索：
1. 训练后提取已知设备的 embedding，取平均作为**原型**（prototype）
2. 推理时计算余弦相似度，低于阈值（如 0.85）→ 标记为 "Unknown"

这个阈值是唯一需要在验证集上调的额外超参数。

---

### 复现顺序建议

1. **先跑通 forward pass**（本文件已验证）
2. **构造合成数据集**测试端到端训练循环
3. **两阶段训练**：先自监督预训练（仅重建 loss），再有监督微调（仅分类 loss 或联合）
4. **接入真实数据集**（如 DeepSig RML2016、ORACLE RF 指纹数据集）
5. **调 `mask_ratio`、`noise_std`、`lambda_recon`** 这三个最关键的超参数